In [1]:
import pandas as pd
import swissparlpy as spp


In [2]:
def get_votes():
    data = spp.get_data('Vote', Language='DE')
    df = pd.DataFrame(data)
    return df

df = get_votes()

In [ ]:
businessNumbers = df["BusinessNumber"].unique()

In [ ]:
spp.get_overview() 

{'MemberParty': ['ID',
  'Language',
  'PartyNumber',
  'PartyName',
  'PersonNumber',
  'PersonIdCode',
  'FirstName',
  'LastName',
  'GenderAsString',
  'PartyFunction',
  'Modified',
  'PartyAbbreviation'],
 'Party': ['ID',
  'Language',
  'PartyNumber',
  'PartyName',
  'StartDate',
  'EndDate',
  'Modified',
  'PartyAbbreviation'],
 'Person': ['ID',
  'Language',
  'PersonNumber',
  'PersonIdCode',
  'Title',
  'TitleText',
  'LastName',
  'GenderAsString',
  'DateOfBirth',
  'DateOfDeath',
  'MaritalStatus',
  'MaritalStatusText',
  'PlaceOfBirthCity',
  'PlaceOfBirthCanton',
  'Modified',
  'FirstName',
  'OfficialName',
  'MilitaryRank',
  'MilitaryRankText',
  'NativeLanguage',
  'NumberOfChildren'],
 'PersonAddress': ['ID',
  'Language',
  'Modified',
  'PersonNumber',
  'AddressType',
  'AddressTypeName',
  'IsPublic',
  'AddressLine1',
  'AddressLine2',
  'AddressLine3',
  'City',
  'CantonName',
  'Comments',
  'CantonNumber',
  'Postcode',
  'CantonAbbreviation'],
 'Pers

In [ ]:
data = spp.get_data("BusinessType", Language="DE")
pd.DataFrame(data)

,ID,BusinessTypeName,BusinessTypeAbbreviation,Language,Modified
0,10,Petition,Pet.,DE,2010-12-26 13:31:48.703000+00:00
1,14,Fragestunde. Frage,Fra.,DE,2010-12-26 13:31:48.703000+00:00
2,5,Motion,Mo.,DE,2010-12-26 13:31:48.703000+00:00
3,3,Standesinitiative,Kt. Iv.,DE,2010-12-26 13:31:48.703000+00:00
4,1,Geschäft des Bundesrates,BRG,DE,2010-12-26 13:31:48.703000+00:00
5,7,Empfehlung,Emp.,DE,2010-12-26 13:31:48.703000+00:00
6,12,Einfache Anfrage,EA,DE,2010-12-26 13:31:48.703000+00:00
7,9,Dringliche Interpellation,D.Ip.,DE,2010-12-26 13:31:48.703000+00:00
8,4,Parlamentarische Initiative,Pa. Iv.,DE,2010-12-26 13:31:48.703000+00:00
9,13,Dringliche Einfache Anfrage,D.EA,DE,2010-12-26 13:31:48.703000+00:00


In [ ]:
opd_client = spp.SwissParlClient(backend="openparldata")
data = opd_client.get_data("votes")[0]

#data = spp.get_data('Vote', Language="DE")
data.get_related_tables()

['voting', 'person', 'person_image', 'bodies']

In [3]:
import os
__location__ = os.path.realpath(os.getcwd())
path = os.path.join(__location__, "voting")


def save_voting_of_vote(id, path):
    if not os.path.exists(path):
        os.mkdir(path)
    data = spp.get_data("Voting", Language="DE", IdVote=id)
    print(f"{data.count} rows loaded.")
    df = pd.DataFrame(data)
    pickle_path = os.path.join(path, f'{id}.pks')
    df.to_pickle(pickle_path)
    print(f"Saved pickle at {pickle_path}")

#votes = get_votes()



In [14]:
votes = spp.get_data('Vote', Language='DE')

i = 0
for vote in votes[100:120]:
    print(f"Loading session {vote['ID']}")
    save_voting_of_vote(vote['ID'], path)
    i = i + 1
    print(i)


df_voting = pd.concat([pd.read_pickle(os.path.join(path, x)) for x in os.listdir(path)])

Loading session 368
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/368.pks
1
Loading session 369
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/369.pks
2
Loading session 382
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/382.pks
3
Loading session 386
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/386.pks
4
Loading session 389
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/389.pks
5
Loading session 392
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/392.pks
6
Loading session 402
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/402.pks
7
Loading session 404
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/404.pks
8
Loading session 411
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/411.pks
9
Loading session 413
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voting/413.pks
10
Loading session 415
200 rows loaded.
Saved pickle at /home/hannah/git/DENG/voti

In [15]:

df_voting = pd.concat([pd.read_pickle(os.path.join(path, x)) for x in os.listdir(path)])

In [16]:
def clean_up_voting(voting):
    voting = voting.reset_index()
    df = voting.drop(["Language", "ParlGroupColour","index"],axis=1)
    return df

df = clean_up_voting(df_voting)

def create_pary_summary(voting):
    mode_df = voting.groupby(['BusinessTitle','IdVote', 'ParlGroupName'])['DecisionText'].agg(lambda x: x.mode().iloc[0]).reset_index()
    counts = pd.crosstab([voting['BusinessTitle'], voting['IdVote'], voting["ParlGroupName"]], voting['DecisionText'])
    vote_values = counts.columns.tolist()
    final_df = pd.merge(mode_df, counts, on=['BusinessTitle', 'IdVote','ParlGroupName'])
    final_df["TotalSeats"] = final_df[vote_values].sum(axis=1)
    final_df.columns = [clean_column(col) for col in final_df.columns]

    return final_df


def clean_column(name):
    import re
    name = str(name)          
    name = name.replace(" ", "_")  
    name = re.sub(r'\W', '', name)
    return name

party_summary = create_pary_summary(df)
party_summary

,BusinessTitle,IdVote,ParlGroupName,DecisionText,Die_Präsidentinder_Präsident_stimmt_nicht,Enthaltung,Hat_nicht_teilgenommen,Ja,Nein,TotalSeats
0,Altlasten. Untersuchungskosten,413,CVP-Fraktion,Ja,0,0,13,14,0,27
1,Altlasten. Untersuchungskosten,413,Die Mitte-Fraktion. Die Mitte. EVP.,Hat nicht teilgenommen,0,0,1,0,0,1
2,Altlasten. Untersuchungskosten,413,EVP/EDU Fraktion,Ja,0,0,2,3,0,5
3,Altlasten. Untersuchungskosten,413,FDP-Liberale Fraktion,Ja,0,0,13,25,1,39
4,Altlasten. Untersuchungskosten,413,Fraktion der Schweizerischen Volkspartei,Ja,1,3,20,30,1,55
...,...,...,...,...,...,...,...,...,...,...
175,Versicherungsaufsicht und Versicherungsvertrag...,389,Fraktion der Schweizerischen Volkspartei,Ja,1,0,14,40,0,55
176,Versicherungsaufsicht und Versicherungsvertrag...,389,Fraktionslos,Nein,0,0,0,2,3,5
177,Versicherungsaufsicht und Versicherungsvertrag...,389,Freisinnig-demokratische Fraktion,Ja,0,0,0,1,0,1
178,Versicherungsaufsicht und Versicherungsvertrag...,389,Grüne Fraktion,Nein,0,2,3,3,7,15


In [ ]:
df_vot

,ID,Language,IdVote,RegistrationNumber,PersonNumber,FirstName,LastName,Canton,CantonName,ParlGroupCode,...,BusinessTitle,BillTitle,IdLegislativePeriod,IdSession,VoteEnd,MeaningYes,MeaningNo,CantonID,Subject,VoteEndWithTimezone
0,8171,DE,389,492,1148,Géraldine,Savary,VD,Waadt,S,...,Versicherungsaufsicht und Versicherungsvertrag...,Bundesgesetz über den Versicherungsvertrag,47,4702,2004-03-17 16:26:09+00:00,,,23,Gesamtabstimmung,2004-03-17 16:26:09+01:00
1,12704,DE,389,492,1129,Silvia,Schenker,BS,Basel-Stadt,S,...,Versicherungsaufsicht und Versicherungsvertrag...,Bundesgesetz über den Versicherungsvertrag,47,4702,2004-03-17 16:26:09+00:00,,,6,Gesamtabstimmung,2004-03-17 16:26:09+01:00
2,24519,DE,389,492,1146,Marianne,Huguenin,VD,Waadt,-,...,Versicherungsaufsicht und Versicherungsvertrag...,Bundesgesetz über den Versicherungsvertrag,47,4702,2004-03-17 16:26:09+00:00,,,23,Gesamtabstimmung,2004-03-17 16:26:09+01:00
3,24520,DE,389,492,1121,Jasmin,Hutter-Hutter,SG,St. Gallen,V,...,Versicherungsaufsicht und Versicherungsvertrag...,Bundesgesetz über den Versicherungsvertrag,47,4702,2004-03-17 16:26:09+00:00,,,16,Gesamtabstimmung,2004-03-17 16:26:09+01:00
4,35905,DE,389,492,1139,Christa,Markwalder,BE,Bern,RL,...,Versicherungsaufsicht und Versicherungsvertrag...,Bundesgesetz über den Versicherungsvertrag,47,4702,2004-03-17 16:26:09+00:00,,,4,Gesamtabstimmung,2004-03-17 16:26:09+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,821955,DE,411,519,138,Jean-Philippe,Maitre,GE,Genf,C,...,Genetische Untersuchungen beim Menschen. Bunde...,Bundesgesetz über genetische Untersuchungen be...,47,4702,2004-03-18 13:38:24+00:00,,,8,Gesamtabstimmung,2004-03-18 13:38:24+01:00
196,821956,DE,411,519,100,Hans Rudolf,Gysin,BL,Basel-Landschaft,RL,...,Genetische Untersuchungen beim Menschen. Bunde...,Bundesgesetz über genetische Untersuchungen be...,47,4702,2004-03-18 13:38:24+00:00,,,5,Gesamtabstimmung,2004-03-18 13:38:24+01:00
197,821957,DE,411,519,172,Paul,Rechsteiner,SG,St. Gallen,S,...,Genetische Untersuchungen beim Menschen. Bunde...,Bundesgesetz über genetische Untersuchungen be...,47,4702,2004-03-18 13:38:24+00:00,,,16,Gesamtabstimmung,2004-03-18 13:38:24+01:00
198,821958,DE,411,519,326,Paul,Günter,BE,Bern,S,...,Genetische Untersuchungen beim Menschen. Bunde...,Bundesgesetz über genetische Untersuchungen be...,47,4702,2004-03-18 13:38:24+00:00,,,4,Gesamtabstimmung,2004-03-18 13:38:24+01:00


In [ ]:
pd.crosstab(df_voting["ID"],df_voting["MeaningNo"])

In [55]:
df_voting[["ParlGroupName", "Decision","DecisionText","BusinessTitle","BillTitle"]]

,ParlGroupName,Decision,DecisionText,BusinessTitle,BillTitle
0,Sozialdemokratische Fraktion,1,Ja,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...
1,Sozialdemokratische Fraktion,1,Ja,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...
2,Fraktionslos,5,Hat nicht teilgenommen,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...
3,Fraktion der Schweizerischen Volkspartei,1,Ja,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...
4,Sozialdemokratische Fraktion,1,Ja,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...
...,...,...,...,...,...
195,Sozialdemokratische Fraktion,3,Enthaltung,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...
196,FDP-Liberale Fraktion,1,Ja,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...
197,CVP-Fraktion,1,Ja,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...
198,Sozialdemokratische Fraktion,3,Enthaltung,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...


In [19]:
df_votes = pd.DataFrame(votes)
df_votes


,ID,Language,RegistrationNumber,BusinessNumber,BusinessShortNumber,BusinessTitle,BusinessAuthor,BillNumber,BillTitle,IdLegislativePeriod,IdSession,SessionName,Subject,MeaningYes,MeaningNo,VoteEnd,VoteEndWithTimezone
0,1,DE,17,20030054,03.054,Stiftung Bibliomedia. Finanzhilfe 2004-2007,,1,Bundesgesetz über die Ausrichtung von Finanzhi...,47,4701,Wintersession 2003,Gesamtabstimmung,,,2003-12-01 18:13:57+00:00,2003-12-01 18:13:57+01:00
1,2,DE,18,20030054,03.054,Stiftung Bibliomedia. Finanzhilfe 2004-2007,,2,Bundesbeschluss betreffend einen Zahlungsrahme...,47,4701,Wintersession 2003,Gesamtabstimmung,,,2003-12-01 18:16:36+00:00,2003-12-01 18:16:36+01:00
2,4,DE,20,20030055,03.055,Verkehrshaus Schweiz. Finanzhilfe 2004-2007,,1,Bundesgesetz über die Ausrichtung von Finanzhi...,47,4701,Wintersession 2003,Gesamtabstimmung,,,2003-12-01 18:35:54+00:00,2003-12-01 18:35:54+01:00
3,5,DE,21,20030055,03.055,Verkehrshaus Schweiz. Finanzhilfe 2004-2007,,2,Bundesbeschluss betreffend einen Zahlungsrahme...,47,4701,Wintersession 2003,Gesamtabstimmung,,,2003-12-01 18:37:29+00:00,2003-12-01 18:37:29+01:00
4,6,DE,22,20030045,03.045,ETH. Leistungsauftrag für die Jahre 2004-2007,,1,Bundesbeschluss über den Leistungsauftrag des ...,47,4701,Wintersession 2003,Gesamtabstimmung,,,2003-12-01 19:01:07+00:00,2003-12-01 19:01:07+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23746,35948,DE,32208,20250066,25.066,Freihandelsabkommen zwischen den EFTA-Staaten ...,,1,Bundesbeschluss über die Genehmigung des Freih...,52,5212,Frühjahrssession 2026,Vote final,Adopter le projet,Rejeter le projet,2026-03-20 09:52:22.229000+00:00,2026-03-20 09:52:22+01:00
23747,35949,DE,32209,20250067,25.067,«Für eine sichere Ernährung – durch Stärkung e...,,1,Bundesbeschluss über die Volksinitiative «Für ...,52,5212,Frühjahrssession 2026,Vote final,Adopter le projet (recommandation de rejeter l...,Rejeter le projet,2026-03-20 09:53:31.476000+00:00,2026-03-20 09:53:31+01:00
23748,35950,DE,32210,20250070,25.070,Freihandelsabkommen zwischen den EFTA-Staaten ...,,1,Bundesbeschluss über die Genehmigung des Freih...,52,5212,Frühjahrssession 2026,Vote final,Adopter le projet,Rejeter le projet,2026-03-20 09:54:24.861000+00:00,2026-03-20 09:54:24+01:00
23749,35951,DE,32211,20250072,25.072,Multilaterale Vereinbarung der zuständigen Beh...,,1,Bundesbeschluss über die Genehmigung der Multi...,52,5212,Frühjahrssession 2026,Vote final,Adopter le projet,Rejeter le projet,2026-03-20 09:55:16.721000+00:00,2026-03-20 09:55:16+01:00


In [ ]:
mode_df = df_voting.groupby(['BusinessTitle','IdVote', 'ParlGroupName'])['DecisionText'].agg(lambda x: x.mode().iloc[0]).reset_index()

mode_df.head(30)

,BusinessTitle,IdVote,ParlGroupName,DecisionText
0,Anerkennung des Völkermordes an den Armeniern ...,143,CVP-Fraktion,Ja
1,Anerkennung des Völkermordes an den Armeniern ...,143,Die Mitte-Fraktion. Die Mitte. EVP.,Hat nicht teilgenommen
2,Anerkennung des Völkermordes an den Armeniern ...,143,EVP/EDU Fraktion,Ja
3,Anerkennung des Völkermordes an den Armeniern ...,143,FDP-Liberale Fraktion,Nein
4,Anerkennung des Völkermordes an den Armeniern ...,143,Fraktion der Schweizerischen Volkspartei,Nein
5,Anerkennung des Völkermordes an den Armeniern ...,143,Fraktionslos,Ja
6,Anerkennung des Völkermordes an den Armeniern ...,143,Freisinnig-demokratische Fraktion,Hat nicht teilgenommen
7,Anerkennung des Völkermordes an den Armeniern ...,143,Grüne Fraktion,Ja
8,Anerkennung des Völkermordes an den Armeniern ...,143,Sozialdemokratische Fraktion,Ja
9,Bericht zur Abschreibung der Motionen Neirynck...,142,CVP-Fraktion,Nein


In [155]:
df_voting = df_voting.reset_index()
df_voting

,index,ID,Language,IdVote,RegistrationNumber,PersonNumber,FirstName,LastName,Canton,CantonName,...,BusinessTitle,BillTitle,IdLegislativePeriod,IdSession,VoteEnd,MeaningYes,MeaningNo,CantonID,Subject,VoteEndWithTimezone
0,0,5583,DE,75,113,1148,Géraldine,Savary,VD,Waadt,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,23,Gesamtabstimmung,2003-12-08 15:25:33+01:00
1,1,11820,DE,75,113,1129,Silvia,Schenker,BS,Basel-Stadt,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,6,Gesamtabstimmung,2003-12-08 15:25:33+01:00
2,2,23037,DE,75,113,1146,Marianne,Huguenin,VD,Waadt,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,23,Gesamtabstimmung,2003-12-08 15:25:33+01:00
3,3,23038,DE,75,113,1121,Jasmin,Hutter-Hutter,SG,St. Gallen,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,16,Gesamtabstimmung,2003-12-08 15:25:33+01:00
4,4,34183,DE,75,113,1147,Margret,Kiener Nellen,BE,Bern,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,4,Gesamtabstimmung,2003-12-08 15:25:33+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,195,808966,DE,67,104,172,Paul,Rechsteiner,SG,St. Gallen,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,16,Gesamtabstimmung,2003-12-09 12:10:33+01:00
9996,196,820110,DE,67,104,100,Hans Rudolf,Gysin,BL,Basel-Landschaft,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,5,Gesamtabstimmung,2003-12-09 12:10:33+01:00
9997,197,820111,DE,67,104,138,Jean-Philippe,Maitre,GE,Genf,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,8,Gesamtabstimmung,2003-12-09 12:10:33+01:00
9998,198,820112,DE,67,104,326,Paul,Günter,BE,Bern,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,4,Gesamtabstimmung,2003-12-09 12:10:33+01:00


In [136]:
counts = pd.crosstab([df_voting['BusinessTitle'], df_voting['IdVote'], df_voting["ParlGroupName"]], df_voting['DecisionText'])
vote_values = counts.columns.tolist()
counts


DecisionText                                                                                        Die Präsidentin/der Präsident stimmt nicht  \
BusinessTitle                                      IdVote ParlGroupName                                                                          
Anerkennung des Völkermordes an den Armeniern i... 143    CVP-Fraktion                                                                       0   
                                                          Die Mitte-Fraktion. Die Mitte. EVP.                                                0   
                                                          EVP/EDU Fraktion                                                                   0   
                                                          FDP-Liberale Fraktion                                                              0   
                                                          Fraktion der Schweizerischen Volkspartei                                           1   
...                                                                                                                                        ...   
Überstellung verurteilter Personen. Änderung de... 34     Fraktion der Schweizerischen Volkspartei                                           1   
                                                          Fraktionslos                                                                       0   
                                                          Freisinnig-demokratische Fraktion                                                  0   
                                                          Grüne Fraktion                                                                     0   
                                                          Sozialdemokratische Fraktion                                                       0   

DecisionText                                                                                        Enthaltung  \
BusinessTitle                                      IdVote ParlGroupName                                          
Anerkennung des Völkermordes an den Armeniern i... 143    CVP-Fraktion                                       7   
                                                          Die Mitte-Fraktion. Die Mitte. EVP.                0   
                                                          EVP/EDU Fraktion                                   0   
                                                          FDP-Liberale Fraktion                              3   
                                                          Fraktion der Schweizerischen Volkspartei           1   
...                                                                                                        ...   
Überstellung verurteilter Personen. Änderung de... 34     Fraktion der Schweizerischen Volkspartei           2   
                                                          Fraktionslos                                       0   
                                                          Freisinnig-demokratische Fraktion                  0   
                                                          Grüne Fraktion                                     0   
                                                          Sozialdemokratische Fraktion                       0   

DecisionText                                                                                        Entschuldigt gemäss Art. 57 Abs. 4  \
BusinessTitle                                      IdVote ParlGroupName                                                                  
Anerkennung des Völkermordes an den Armeniern i... 143    CVP-Fraktion                                                               0   
                                                          Die Mitte-Fraktion. Die Mitte. EVP.                                        0   
                                                          EVP/EDU 

In [129]:
final_df = pd.merge(mode_df, counts, on=['BusinessTitle', 'IdVote','ParlGroupName'])

In [140]:
final_df

final_df["TotalSeats"] = final_df[vote_values].sum(axis=1)

In [141]:
final_df

,BusinessTitle,IdVote,ParlGroupName,DecisionText,Die Präsidentin/der Präsident stimmt nicht,Enthaltung,Entschuldigt gemäss Art. 57 Abs. 4,Hat nicht teilgenommen,Ja,Nein,TotalSeats
0,Anerkennung des Völkermordes an den Armeniern ...,143,CVP-Fraktion,Ja,0,7,0,2,18,0,27
1,Anerkennung des Völkermordes an den Armeniern ...,143,Die Mitte-Fraktion. Die Mitte. EVP.,Hat nicht teilgenommen,0,0,0,1,0,0,1
2,Anerkennung des Völkermordes an den Armeniern ...,143,EVP/EDU Fraktion,Ja,0,0,0,1,4,0,5
3,Anerkennung des Völkermordes an den Armeniern ...,143,FDP-Liberale Fraktion,Nein,0,3,0,2,8,26,39
4,Anerkennung des Völkermordes an den Armeniern ...,143,Fraktion der Schweizerischen Volkspartei,Nein,1,1,0,4,7,41,54
...,...,...,...,...,...,...,...,...,...,...,...
429,Überstellung verurteilter Personen. Änderung d...,34,Fraktion der Schweizerischen Volkspartei,Ja,1,2,0,19,33,0,55
430,Überstellung verurteilter Personen. Änderung d...,34,Fraktionslos,Ja,0,0,0,2,3,0,5
431,Überstellung verurteilter Personen. Änderung d...,34,Freisinnig-demokratische Fraktion,Ja,0,0,0,12,28,0,40
432,Überstellung verurteilter Personen. Änderung d...,34,Grüne Fraktion,Ja,0,0,0,5,10,0,15


In [88]:
df_voting[df_voting["DecisionText"] == 'Ja']

,ID,Language,IdVote,RegistrationNumber,PersonNumber,FirstName,LastName,Canton,CantonName,ParlGroupCode,...,BillTitle,IdLegislativePeriod,IdSession,VoteEnd,MeaningYes,MeaningNo,CantonID,Subject,VoteEndWithTimezone,DecisionCount
0,5583,DE,75,113,1148,Géraldine,Savary,VD,Waadt,S,...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,23,Gesamtabstimmung,2003-12-08 15:25:33+01:00,29
1,11820,DE,75,113,1129,Silvia,Schenker,BS,Basel-Stadt,S,...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,6,Gesamtabstimmung,2003-12-08 15:25:33+01:00,29
3,23038,DE,75,113,1121,Jasmin,Hutter-Hutter,SG,St. Gallen,V,...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,16,Gesamtabstimmung,2003-12-08 15:25:33+01:00,34
4,34183,DE,75,113,1147,Margret,Kiener Nellen,BE,Bern,S,...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,4,Gesamtabstimmung,2003-12-08 15:25:33+01:00,29
5,34184,DE,75,113,1139,Christa,Markwalder,BE,Bern,RL,...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,4,Gesamtabstimmung,2003-12-08 15:25:33+01:00,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,808964,DE,67,104,34,Gerold,Bührer,SH,Schaffhausen,RL,...,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,17,Gesamtabstimmung,2003-12-09 12:10:33+01:00,34
194,808965,DE,67,104,14,Duri,Bezzola,GR,Graubünden,RL,...,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,10,Gesamtabstimmung,2003-12-09 12:10:33+01:00,34
196,820110,DE,67,104,100,Hans Rudolf,Gysin,BL,Basel-Landschaft,RL,...,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,5,Gesamtabstimmung,2003-12-09 12:10:33+01:00,34
197,820111,DE,67,104,138,Jean-Philippe,Maitre,GE,Genf,C,...,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,8,Gesamtabstimmung,2003-12-09 12:10:33+01:00,22


In [ ]:


df_voting.drop("Language",axis=1)
df_votes.drop("Language",axis=1)



,ID,IdVote,RegistrationNumber,PersonNumber,FirstName,LastName,Canton,CantonName,ParlGroupCode,ParlGroupColour,...,BusinessTitle,BillTitle,IdLegislativePeriod,IdSession,VoteEnd,MeaningYes,MeaningNo,CantonID,Subject,VoteEndWithTimezone
0,5583,75,113,1148,Géraldine,Savary,VD,Waadt,S,#FFFF0000,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,23,Gesamtabstimmung,2003-12-08 15:25:33+01:00
1,11820,75,113,1129,Silvia,Schenker,BS,Basel-Stadt,S,#FFFF0000,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,6,Gesamtabstimmung,2003-12-08 15:25:33+01:00
2,23037,75,113,1146,Marianne,Huguenin,VD,Waadt,-,#FFF0E68C,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,23,Gesamtabstimmung,2003-12-08 15:25:33+01:00
3,23038,75,113,1121,Jasmin,Hutter-Hutter,SG,St. Gallen,V,#FF006400,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,16,Gesamtabstimmung,2003-12-08 15:25:33+01:00
4,34183,75,113,1147,Margret,Kiener Nellen,BE,Bern,S,#FFFF0000,...,Freihandelsabkommen zwischen den EFTA-Staaten ...,Bundesbeschluss zum Freihandelsabkommen zwisch...,47,4701,2003-12-08 15:25:33+00:00,,,4,Gesamtabstimmung,2003-12-08 15:25:33+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,808966,67,104,172,Paul,Rechsteiner,SG,St. Gallen,S,#FFFF0000,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,16,Gesamtabstimmung,2003-12-09 12:10:33+01:00
196,820110,67,104,100,Hans Rudolf,Gysin,BL,Basel-Landschaft,RL,#FF00BFFF,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,5,Gesamtabstimmung,2003-12-09 12:10:33+01:00
197,820111,67,104,138,Jean-Philippe,Maitre,GE,Genf,C,#FFFFA500,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,8,Gesamtabstimmung,2003-12-09 12:10:33+01:00
198,820112,67,104,326,Paul,Günter,BE,Bern,S,#FFFF0000,...,Voranschlag 2004,Bundesbeschluss I über den Voranschlag für das...,47,4701,2003-12-09 12:10:33+00:00,,,4,Gesamtabstimmung,2003-12-09 12:10:33+01:00


In [162]:
data = spp.get_data('Business', Language='DE')
df = pd.DataFrame(data)
df


,ID,Language,BusinessShortNumber,BusinessType,BusinessTypeName,BusinessTypeAbbreviation,Title,Description,InitialSituation,Proceedings,...,SubmissionCouncilAbbreviation,SubmissionSession,SubmissionLegislativePeriod,FirstCouncil1,FirstCouncil1Name,FirstCouncil1Abbreviation,FirstCouncil2,FirstCouncil2Name,FirstCouncil2Abbreviation,TagNames
0,19780222,DE,78.222,4,Parlamentarische Initiative,Pa. Iv.,Strafgesetzbuch. Schwangerschaftsabbruch (Gira...,Bericht der Kommission vom 27.08.1979,,,...,NR,4311,40,1,Nationalrat,NR,0,,,
1,19800226,DE,80.226,4,Parlamentarische Initiative,Pa. Iv.,BV. Unternehmensrecht (Jelmini),,,,...,NR,4311,41,1,Nationalrat,NR,0,,,
2,19850019,DE,85.019,1,Geschäft des Bundesrates,BRG,Friedliche Nutzung der Kernenergie. Abkommen m...,Botschaft und Beschlussesentwurf vom 1. Mai 19...,<p>Das Abkommen setzt den öffentlich-rechtlich...,<p></p><p>Beide Räte beschlossen diskussionslo...,...,,4311,42,1,Nationalrat,NR,0,,,
3,19850227,DE,85.227,4,Parlamentarische Initiative,Pa. Iv.,Sozialversicherungsrecht,,<p>Das Schweizer Sozialversicherungsrecht ist ...,<p></p><p>Der <b>Ständerat</b> stimmte den Ant...,...,SR,4311,42,2,Ständerat,SR,0,,,
4,19870069,DE,87.069,1,Geschäft des Bundesrates,BRG,Eisenbahngesetz. Aenderung,Botschaft und Gesetzesentwürfe vom 18. Novembe...,,,...,,4311,42,2,Ständerat,SR,0,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66344,20269013,DE,26.9013,2,Geschäft des Parlaments,PAG,Ausserordentliche Session. Strafverfolgung,,,,...,,5212,52,0,,,0,,,
66345,20269014,DE,26.9014,2,Geschäft des Parlaments,PAG,Gedenkmoment für die Betroffenen der Ereigniss...,,,,...,,5212,52,0,,,0,,,
66346,20269015,DE,26.9015,2,Geschäft des Parlaments,PAG,Ausserordentliche Session. Mieten,,,,...,,5212,52,0,,,0,,,
66347,20269016,DE,26.9016,2,Geschäft des Parlaments,PAG,Aktuelle Debatte «Gift und Gewässerschutz»,,,,...,,5212,52,0,,,0,,,
